In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix, vstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from sklearn.decomposition import TruncatedSVD
import pickle
from pathlib import Path
from joblib import Parallel, delayed
import numpy as np
import pandas as pd
import math
import pickle
from collections import defaultdict
from pathlib import Path
import numpy as np, pandas as pd, pickle
from pathlib import Path
from scipy.sparse import csr_matrix
from sklearn.preprocessing import normalize
import matplotlib.pyplot as plt
import numpy as np

### Идея работы рекомендательной системы
У системы будет два слоя: контентный слой и коллабаративный слой.
В контентном слое система будет искать игры, похожие на те, в которые игрок уже играл, основываяс на отзывах игроков и изданий, описаний игр, жанров игр. Таким образом система найдёт 300 игр которые могут понравиться игроку.
Коллаборативный слой отберёт из этих 300 игр 10 с помощью itembased и userdased. Скор с контентного слоя тоже будет учитываться.

Контентный слой решит проблему холодного входа и облегчит item/userbased, так как мы будем искать соседей не по всему набору игр, а по 300 отобранным. 

### EDA + работа с датасетом
Убираем из датасета информацию о платформе на которой игрок проходил игру(если игрок проходил на нескольких платформах, то оставляем отзыв с максимальной оценкой), дату прохождения, а также информацию от изданий.

In [2]:
reviews_df = pd.read_csv('mgames/games_reviews.csv')
games_df   = pd.read_csv('mgames/games.csv')

print("=== ДО очистки ===")
print(f"reviews: {reviews_df.shape}, games: {games_df.shape}")
print(f"Колонки reviews: {list(reviews_df.columns)}")
print(f"Колонки games:   {list(games_df.columns)}")


reviews_df['id'] = reviews_df['id'].astype(int)
games_df['id']   = games_df['id'].astype(int)


print("\nreview_type до фильтрации:")
print(reviews_df['review_type'].value_counts(dropna=False))

reviews_df = reviews_df[reviews_df['review_type'] == 'user'].copy()

print("\nreview_type после фильтрации:")
print(reviews_df['review_type'].value_counts(dropna=False))

reviews_df = reviews_df.drop(
    columns=['review_type', 'platform', 'date', 'publicationName'],
    errors='ignore'
)

reviews_df['score'] = pd.to_numeric(reviews_df['score'], errors='coerce')

key_cols = ['author', 'id', 'quote', 'score']
reviews_df = reviews_df.dropna(subset=key_cols)

reviews_df['quote'] = reviews_df['quote'].astype(str).str.strip()
reviews_df = reviews_df[reviews_df['quote'].str.len() >= 5]

print("\n=== Распределение rating ===")
print(reviews_df['score'].describe())
print("\nУникальные значения score:")
print(sorted(reviews_df['score'].unique()))

before = len(reviews_df)
reviews_df = reviews_df.drop_duplicates(subset=['author', 'id'], keep='last')
print(f"\nУдалено дубликатов (author, id): {before - len(reviews_df)}")

print("\n=== ПОСЛЕ очистки ===")
n_users = reviews_df['author'].nunique()
n_games = reviews_df['id'].nunique()
n_reviews = len(reviews_df)

print(f"Юзеров:  {n_users}")
print(f"Игр:     {n_games}")
print(f"Отзывов: {n_reviews}")
print(f"Плотность: {n_reviews / (n_users * n_games):.6f}")

=== ДО очистки ===
reviews: (1656049, 9), games: (13442, 16)
Колонки reviews: ['id', 'title', 'quote', 'score', 'date', 'platform', 'author', 'publicationName', 'review_type']
Колонки games:   ['id', 'title', 'releaseDate', 'rating', 'genres', 'description', 'platforms', 'metascore', 'metascore_count', 'metascore_sentiment', 'userscore', 'userscore_count', 'userscore_sentiment', 'platform_metascores', 'developer', 'publisher']

review_type до фильтрации:
review_type
user      1131692
critic     524357
Name: count, dtype: int64

review_type после фильтрации:
review_type
user    1131692
Name: count, dtype: int64

=== Распределение rating ===
count    1.130798e+06
mean     6.777908e+01
std      3.625846e+01
min      0.000000e+00
25%      4.000000e+01
50%      8.000000e+01
75%      1.000000e+02
max      1.000000e+02
Name: score, dtype: float64

Уникальные значения score:
[np.float64(0.0), np.float64(10.0), np.float64(20.0), np.float64(30.0), np.float64(40.0), np.float64(50.0), np.float64(6

In [3]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42


def proportional_split(df, test_size=0.2, min_test=1, seed=42):
    rng = np.random.RandomState(seed)
    df = df.copy()
    df['_rand'] = rng.rand(len(df))
    df = df.sort_values(['author', '_rand'])

    counts = df.groupby('author')['id'].transform('size')
    df['_rank'] = df.groupby('author').cumcount(ascending=False)

    target_test = np.maximum(min_test, (counts * test_size).round().astype(int))
    target_test = np.minimum(target_test, counts - 1)

    test_mask = (df['_rank'] < target_test) & (counts > min_test)

    test_df  = df[test_mask].drop(columns=['_rand', '_rank']).reset_index(drop=True)
    train_df = df[~test_mask].drop(columns=['_rand', '_rank']).reset_index(drop=True)

    return train_df, test_df

train_df, test_df = proportional_split(reviews_df, test_size=0.2,
                                        min_test=1, seed=RANDOM_STATE)


print("=== SPLIT ===")
print(f"train: {len(train_df):>8d} отзывов | "
      f"{train_df['author'].nunique():>6d} юзеров | "
      f"{train_df['id'].nunique():>6d} игр")
print(f"test:  {len(test_df):>8d} отзывов | "
      f"{test_df['author'].nunique():>6d} юзеров | "
      f"{test_df['id'].nunique():>6d} игр")


common_users = set(train_df['author']) & set(test_df['author'])
print(f"Юзеров в обоих наборах: {len(common_users)}")


test_only_users = set(test_df['author']) - set(train_df['author'])
print(f"Юзеров только в test:   {len(test_only_users)}")

test_counts = test_df.groupby('author').size()
print("\n=== Распределение test-отзывов на юзера ===")
print(test_counts.value_counts().sort_index())

=== SPLIT ===
train:   889593 отзывов | 475976 юзеров |  11822 игр
test:    198600 отзывов | 137492 юзеров |   9224 игр
Юзеров в обоих наборах: 137492
Юзеров только в test:   0

=== Распределение test-отзывов на юзера ===
1      118644
2        9360
3        3696
4        1754
5        1063
        ...  
130         1
139         1
153         1
192         1
261         1
Name: count, Length: 83, dtype: int64


### Content_based
Будем использовать TF-IDF на отзывах игроков и изданий и описаниях игр, чтобы вычленить из них ключевые слова, и впоследствии сравнивать игры на похожесть по этим словам.
Положительные и негативные отзывы будем рассмаривать отдельно, чтобы разделять сильные и слабые стороны игр.

In [4]:
RANDOM_STATE = 42
POS_THRESHOLD = 70
NEG_THRESHOLD = 60
MIN_REVIEW_LEN = 10          
MIN_DF = 3                   
MIN_DF_NEG = 2               
MAX_DF = 0.5
NGRAM_RANGE = (1, 2)
MAX_FEATURES = 20000
OUTPUT_DIR = Path('artifacts')
OUTPUT_DIR.mkdir(exist_ok=True)


def join_reviews_by_game(df, min_len=MIN_REVIEW_LEN):
    df = df[df['quote'].str.len() >= min_len].copy()
    texts = (
        df.groupby('id')['quote']
        .apply(lambda x: ' '.join(x.astype(str)))
    )
    return texts


pos_texts = join_reviews_by_game(train_df[train_df['score'] >= POS_THRESHOLD])
neg_texts = join_reviews_by_game(train_df[train_df['score'] <= NEG_THRESHOLD])

print(f"Игр с позитивными отзывами: {len(pos_texts)}")
print(f"Игр с негативными отзывами: {len(neg_texts)}")


games_indexed = games_df.set_index('id') 
game_descriptions = games_indexed['description'].fillna('').astype(str)

print(f"Игр в games_df: {len(game_descriptions)}")
print(f"С непустым описанием: {(game_descriptions.str.len() > 0).sum()}")

all_games = sorted(
    set(game_descriptions.index) | set(pos_texts.index) | set(neg_texts.index)
)
print(f"Всего игр в content-слое: {len(all_games)}")

descs_full = game_descriptions.reindex(all_games, fill_value='')
pos_full = pos_texts.reindex(all_games, fill_value='')
neg_full = neg_texts.reindex(all_games, fill_value='')

pos_corpus = (descs_full + ' ' + pos_full).str.strip()
neg_corpus = (descs_full + ' ' + neg_full).str.strip()

print(f"Игр с непустым pos-текстом: {(pos_corpus.str.len() > 0).sum()}")
print(f"Игр с непустым neg-текстом: {(neg_corpus.str.len() > 0).sum()}")


Игр с позитивными отзывами: 11112
Игр с негативными отзывами: 9498
Игр в games_df: 13442
С непустым описанием: 13394
Всего игр в content-слое: 13451
Игр с непустым pos-текстом: 13430
Игр с непустым neg-текстом: 13414


In [5]:
tfidf_pos = TfidfVectorizer(
    min_df=MIN_DF,
    max_df=MAX_DF,
    ngram_range=NGRAM_RANGE,
    sublinear_tf=True,
    stop_words='english',
    max_features=MAX_FEATURES,
    strip_accents='unicode',
    lowercase=True,
)

tfidf_neg = TfidfVectorizer(
    min_df=MIN_DF_NEG,
    max_df=MAX_DF,
    ngram_range=NGRAM_RANGE,
    sublinear_tf=True,
    stop_words='english',
    max_features=MAX_FEATURES,
    strip_accents='unicode',
    lowercase=True,
)

X_pos = tfidf_pos.fit_transform(pos_corpus)
X_neg = tfidf_neg.fit_transform(neg_corpus)

X_neg_signed = X_neg.copy()
X_neg_signed.data = X_neg_signed.data * -1.0

X_content = hstack([X_pos, X_neg_signed]).tocsr().astype(np.float32)



X_content_norm = normalize(X_content, norm='l2', axis=1).astype(np.float32)


game_to_idx_content = {g: i for i, g in enumerate(all_games)}
idx_to_game_content = {i: g for g, i in game_to_idx_content.items()}



artifacts = {
    'X_content_norm':       X_content_norm,
    'X_pos':                X_pos,
    'X_neg':                X_neg,
    'all_games':            all_games,
    'game_to_idx_content':  game_to_idx_content,
    'idx_to_game_content':  idx_to_game_content,   # ← добавили
    'tfidf_pos':            tfidf_pos,
    'tfidf_neg':            tfidf_neg,
}

with open(OUTPUT_DIR / 'content_layer.pkl', 'wb') as f:
    pickle.dump(artifacts, f, protocol=4)

### Item/User based
Итоговый скор для игры будем считать по item/userbased и по контентному скору

In [6]:
MIN_USER_REVIEWS = 5
MIN_GAME_REVIEWS = 20
LIKED_THRESHOLD  = 70
N_CONTENT_CANDS  = 300
TOP_K_SIM_USERS  = 50
ALPHA_ITEM       = 0.5
ALPHA_USER       = 0.5
RANDOM_STATE     = 42


def build_ui_pairs(reviews_df):
    df = reviews_df.copy()
    df['score'] = pd.to_numeric(df['score'], errors='coerce')
    df = df.dropna(subset=['score', 'author', 'id'])
    return df.groupby(['author', 'id'], as_index=False)['score'].mean()

ui_df = build_ui_pairs(train_df)
print(f"UI-пар: {len(ui_df)}, юзеров: {ui_df['author'].nunique()}, "
      f"игр: {ui_df['id'].nunique()}")


user_counts = ui_df.groupby('author')['id'].count()
game_counts = ui_df.groupby('id')['author'].count()

hot_users = np.array(sorted(user_counts[user_counts > MIN_USER_REVIEWS].index))
hot_games = np.array(sorted(game_counts[game_counts > MIN_GAME_REVIEWS].index))
print(f"Hot users: {len(hot_users)}, hot games: {len(hot_games)}")


all_rated_games = np.array(sorted(ui_df['id'].unique()))
game_to_row = {g: i for i, g in enumerate(all_rated_games)}
user_to_col = {u: i for i, u in enumerate(hot_users)}

ui_hu = ui_df[ui_df['author'].isin(hot_users)]
rows = ui_hu['id'].map(game_to_row).values
cols = ui_hu['author'].map(user_to_col).values
vals = ui_hu['score'].values.astype(np.float32)

R_items = csr_matrix((vals, (rows, cols)),
                     shape=(len(all_rated_games), len(hot_users)),
                     dtype=np.float32)

item_nnz  = np.diff(R_items.indptr)
item_sums = np.asarray(R_items.sum(axis=1)).ravel()
item_means = item_sums / np.maximum(item_nnz, 1)

R_items_c = R_items.copy()
R_items_c.data = R_items_c.data - np.repeat(item_means, item_nnz)
R_items_norm = normalize(R_items_c, norm='l2', axis=1)

game_to_col_hg = {g: i for i, g in enumerate(hot_games)}

ui_hg = ui_df[ui_df['id'].isin(hot_games) & ui_df['author'].isin(hot_users)]
rows = ui_hg['author'].map(user_to_col).values
cols = ui_hg['id'].map(game_to_col_hg).values
vals = ui_hg['score'].values.astype(np.float32)

R_users = csr_matrix((vals, (rows, cols)),
                     shape=(len(hot_users), len(hot_games)),
                     dtype=np.float32)

user_nnz = np.diff(R_users.indptr)
user_sums = np.asarray(R_users.sum(axis=1)).ravel()
user_means = user_sums / np.maximum(user_nnz, 1)

R_users_c = R_users.copy()
R_users_c.data = R_users_c.data - np.repeat(user_means, user_nnz)
R_users_norm = normalize(R_users_c, norm='l2', axis=1)


def content_candidates_with_scores(user_ratings, content_artifacts, n_candidates):
    """
    То же, что content_candidates, но возвращает и cosine-скоры.
    Возврат: (list[game_id], np.ndarray scores той же длины)
    """
    X    = content_artifacts['X_content_norm']
    g2i  = content_artifacts['game_to_idx_content']
    i2g  = content_artifacts['idx_to_game_content']
    n_games = X.shape[0]

    liked = user_ratings.loc[user_ratings['score'] >= LIKED_THRESHOLD, 'id'].tolist()
    liked_idx = [g2i[g] for g in liked if g in g2i]
    if not liked_idx:
        return [], np.array([])

    profile = np.asarray(X[liked_idx].mean(axis=0)).ravel()
    scores  = np.asarray(X.dot(profile)).ravel()

    rated_idx = [g2i[g] for g in user_ratings['id'] if g in g2i]
    scores[rated_idx] = -np.inf

    n_take = min(n_candidates, n_games)
    top = np.argpartition(-scores, n_take - 1)[:n_take]
    top = top[np.argsort(-scores[top])]
    top = [i for i in top if np.isfinite(scores[i])]

    return [i2g[i] for i in top], scores[top]


def item_based_scores(user_ratings, candidates):
    user_mean = user_ratings['score'].mean()
    rated = user_ratings.copy()
    rated['centered'] = rated['score'] - user_mean
    rated_g = [g for g in rated['id'] if g in game_to_row]
    rated_c = rated.set_index('id').loc[rated_g, 'centered'].values
    cand_g  = [g for g in candidates if g in game_to_row]

    out = {g: 0.0 for g in candidates}
    if not cand_g or not rated_g:
        return out

    C = R_items_norm[[game_to_row[g] for g in cand_g]]
    L = R_items_norm[[game_to_row[g] for g in rated_g]]
    S = np.asarray((C @ L.T).todense())
    num = S @ rated_c
    den = np.abs(S).sum(axis=1)

    scores = _safe_divide(num, den)
    for j, g in enumerate(cand_g):
        out[g] = float(scores[j])
    return out


def _user_vector(user_ratings):
    hr = user_ratings[user_ratings['id'].isin(hot_games)]
    if len(hr) == 0:
        return None
    v = np.zeros(len(hot_games), dtype=np.float32)
    cols = hr['id'].map(game_to_col_hg).values
    vals = hr['score'].values.astype(np.float32)
    v[cols] = vals - vals.mean()
    nrm = np.linalg.norm(v)
    return v / nrm if nrm > 0 else None


def user_based_scores(user_ratings, candidates, top_k=TOP_K_SIM_USERS):
    u_vec = _user_vector(user_ratings)
    out = {g: 0.0 for g in candidates}
    if u_vec is None:
        return out

    sims = np.asarray(R_users_norm @ u_vec).ravel()
    k = min(top_k, len(sims))
    top_users = np.argpartition(-sims, k - 1)[:k]
    top_sims  = sims[top_users]

    cand_g = [g for g in candidates if g in game_to_col_hg]
    if not cand_g:
        return out
    cand_cols = [game_to_col_hg[g] for g in cand_g]

    sub = np.asarray(R_users[top_users][:, cand_cols].todense())
    mask = (sub != 0).astype(np.float32)
    sub_c = (sub - user_means[top_users][:, None]) * mask

    num = top_sims @ sub_c
    den = np.abs(top_sims) @ mask

    scores = _safe_divide(num, den)
    for j, g in enumerate(cand_g):
        out[g] = float(scores[j])
    return out

def _safe_divide(num, den, fill=0.0):
    out = np.full_like(num, fill, dtype=np.float64)
    np.divide(num, den, out=out, where=(den != 0))
    return out

def _minmax(x):
    lo, hi = x.min(), x.max()
    return (x - lo) / (hi - lo) if hi > lo else np.zeros_like(x)

def _safe_zscore(x):
    x = np.asarray(x, dtype=np.float64)
    std = x.std()
    if std < 1e-9:
        return np.zeros_like(x)
    return (x - x.mean()) / std

def _is_alive(x, min_nnz_frac=0.05):
    return 1.0 if (np.abs(x) > 1e-9).mean() >= min_nnz_frac else 0.0

def _rrf_fuse(ranked_lists, k=60, n=None):

    out = np.zeros(n, dtype=np.float64)
    for w, order in ranked_lists:
        for pos, idx in enumerate(order):
            out[idx] += w * 1.0 / (k + pos + 1)
    return out


def recommend_for_user(user_id, ui_df, content_artifacts,
                       n_top=10, n_candidates=N_CONTENT_CANDS,
                       alpha_content=0.5,
                       alpha_item=0.3,
                       alpha_user=0.2,
                       top_k_sim_users=TOP_K_SIM_USERS,
                       exclude_ids=None,
                       use_rrf=False,
                       rrf_k=60,
                       verbose=False):

    user_ratings = ui_df[ui_df['author'] == user_id]
    if len(user_ratings) == 0:
        return []

    cands, c_arr = content_candidates_with_scores(
        user_ratings, content_artifacts, n_candidates
    )
    if exclude_ids:
        keep = [i for i, g in enumerate(cands) if g not in exclude_ids]
        cands = [cands[i] for i in keep]
        c_arr = c_arr[keep] if len(c_arr) else c_arr
    if not cands:
        return []

    item_s = item_based_scores(user_ratings, cands)
    user_s = user_based_scores(user_ratings, cands, top_k=top_k_sim_users)

    i_arr = np.array([item_s[g] for g in cands], dtype=np.float64)
    u_arr = np.array([user_s[g] for g in cands], dtype=np.float64)

    if use_rrf:
        final = _rrf_fuse([
            (alpha_content, np.argsort(-c_arr)),
            (alpha_item,    np.argsort(-i_arr)),
            (alpha_user,    np.argsort(-u_arr)),
        ], k=rrf_k, n=len(cands))
    else:
        c_norm = _safe_zscore(c_arr)
        i_norm = _safe_zscore(i_arr) * _is_alive(i_arr)
        u_norm = _safe_zscore(u_arr) * _is_alive(u_arr)
        final  = alpha_content * c_norm + alpha_item * i_norm + alpha_user * u_norm

    if verbose:
        print(f"content nnz={(_is_alive(c_arr)):.0f} "
              f"item nnz={np.mean(np.abs(i_arr)>1e-9):.2f} "
              f"user nnz={np.mean(np.abs(u_arr)>1e-9):.2f}")

    order = np.argsort(-final)[:n_top]
    return [(cands[i], float(final[i])) for i in order]

UI-пар: 889593, юзеров: 475976, игр: 11822
Hot users: 22777, hot games: 3970


### Итоговая система

In [7]:
K_EVAL            = 10
LIKED_THRESHOLD   = 70 
OUTPUT_DIR        = Path('artifacts')


def precision_at_k(recs, relevant, k):
    if not relevant or not recs:
        return 0.0
    return len(set(recs[:k]) & relevant) / k


def hit_rate_at_k(recs, relevant, k):
    if not relevant or not recs:
        return 0.0
    return 1.0 if any(g in relevant for g in recs[:k]) else 0.0


def mrr_at_k(recs, relevant, k):
    if not relevant or not recs:
        return 0.0
    for i, g in enumerate(recs[:k]):
        if g in relevant:
            return 1.0 / (i + 1)
    return 0.0


def ndcg_at_k(recs, relevant, k):
    rels = [1 if g in relevant else 0 for g in recs[:k]]
    if not rels:
        return 0.0
    dcg  = sum(r / math.log2(i + 2) for i, r in enumerate(rels))
    ideal = sorted(rels, reverse=True)
    idcg = sum(r / math.log2(i + 2) for i, r in enumerate(ideal))
    return dcg / idcg if idcg > 0 else 0.0


def build_ground_truth(test_df, user_to_idx, min_rating=LIKED_THRESHOLD):
    df = test_df
    if min_rating is not None:
        df = df[df['score'] >= min_rating]

    gt = defaultdict(set)
    for author, game_id in zip(df['author'], df['id']):
        u = user_to_idx.get(author)
        if u is not None:
            gt[u].add(game_id)
    return dict(gt)


def make_recommend_fn(ui_df,
                      content_artifacts,
                      idx_to_user,
                      n_candidates=300,
                      alpha_item=0.5,
                      alpha_user=0.5,
                      top_k_sim_users=50):
    train_rated = ui_df.groupby('author')['id'].apply(set).to_dict()

    def _fn(user_idx, k):
        user_id = idx_to_user.get(user_idx)
        if user_id is None:
            return []
        user_ratings = ui_df[ui_df['author'] == user_id]
        if user_ratings.empty:
            return []
        excl = train_rated.get(user_id, set())

        recs = recommend_for_user(
            user_id, ui_df, content_artifacts,
            n_top=k,
            n_candidates=n_candidates,
            alpha_item=alpha_item,
            alpha_user=alpha_user,
            top_k_sim_users=top_k_sim_users,
            exclude_ids=excl,
        )
        return [g for g, _ in recs]

    return _fn

In [9]:
def evaluate_users_parallel(recommend_fn, user_indices, ground_truth,
                            k=10, n_jobs=-1):
    def _one(u):
        relevant = ground_truth.get(u, set())
        if not relevant:
            return None
        recs = recommend_fn(u, k)
        return (precision_at_k(recs, relevant, k),
                hit_rate_at_k(recs, relevant, k),
                mrr_at_k(recs, relevant, k),
                ndcg_at_k(recs, relevant, k))

    out = Parallel(n_jobs=n_jobs, batch_size=64)(
        delayed(_one)(u) for u in user_indices
    )
    out = [r for r in out if r is not None]
    if not out:
        return {f'precision@{k}': 0.0, f'hit_rate@{k}': 0.0,
                f'mrr@{k}': 0.0, f'ndcg@{k}': 0.0, 'n_users': 0}
    arr = np.array(out)
    return {
        f'precision@{k}': float(arr[:, 0].mean()),
        f'hit_rate@{k}':  float(arr[:, 1].mean()),
        f'mrr@{k}':       float(arr[:, 2].mean()),
        f'ndcg@{k}':      float(arr[:, 3].mean()),
        'n_users':        len(out),
    }



def evaluate_stratified(recommend_fn, test_df, user_to_idx,
                        k=10, min_rating=LIKED_THRESHOLD, verbose=True):
    """
    Страты по числу relevant у юзера:
        cold (1) | warm (2-4) | active (5+) | all
    """
    ground_truth = build_ground_truth(test_df, user_to_idx, min_rating)
    user_rel_count = {u: len(rel) for u, rel in ground_truth.items()}

    strata = {
        'cold (1)':    [u for u, c in user_rel_count.items() if c == 1],
        'warm (2-4)':  [u for u, c in user_rel_count.items() if 2 <= c <= 4],
        'active (5+)': [u for u, c in user_rel_count.items() if c >= 5],
        'all':         list(user_rel_count.keys()),
    }

    rows = []
    for name, users in strata.items():
        if not users:
            continue

        res = evaluate_users_parallel(recommend_fn, users, ground_truth, k=k)
        res['stratum'] = name
        rows.append(res)

    df = pd.DataFrame(rows).set_index('stratum')
    cols = [c for c in df.columns if c != 'n_users'] + ['n_users']
    return df[cols]


with open(OUTPUT_DIR / 'content_layer.pkl', 'rb') as f:
    content_artifacts = pickle.load(f)


all_users = sorted(set(ui_df['author']) | set(test_df['author']))
user_to_idx = {u: i for i, u in enumerate(all_users)}
idx_to_user = {i: u for u, i in user_to_idx.items()}

rec_fn = make_recommend_fn(
    ui_df, content_artifacts, idx_to_user,
    n_candidates=300, alpha_item=0.5, alpha_user=0.5, top_k_sim_users=50,
)

report = evaluate_stratified(rec_fn, test_df, user_to_idx, k=10)
print(report)

             precision@10  hit_rate@10    mrr@10   ndcg@10  n_users
stratum                                                            
cold (1)         0.013211     0.132115  0.055676  0.073435    83140
warm (2-4)       0.022970     0.204382  0.073091  0.102995    10270
active (5+)      0.054812     0.404717  0.157476  0.212131     2629
all              0.015394     0.147305  0.060325  0.080393    96039
